In [1]:
# Install required libraries for RAG pipeline
!pip install langchain langchain-community langchain-core
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf2
!pip install google-generativeai
!pip install tiktoken
!pip install langchain-text-splitters
!pip install requests
!pip install google-genai
!pip install -U `langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 885.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
import sys
!{sys.executable} -m pip install -U 'langchain-huggingface'

# Importing all the libraries required
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import SentenceTransformersTokenTextSplitter
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import GooglePalm
from langchain_classic.chains import RetrievalQA
import chromadb
import requests
import os
import tempfile

/tmp/ipykernel_3765/4099384689.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
# Extraction of text from PDF file
def extract_pdf_content(pdf_path):
  text = ""
  # Download the PDF if the path is a URL
  if pdf_path.startswith("http://") or pdf_path.startswith("https://"):
    response = requests.get(pdf_path)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    # Create a temporary file to save the PDF content
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_pdf:
      temp_pdf.write(response.content)
      temp_pdf_path = temp_pdf.name

    try:
      with open(temp_pdf_path, "rb") as pdf_file:
        pdf_reader = PdfReader(pdf_file)
        for page in pdf_reader.pages:
          text += page.extract_text()
    finally:
      os.remove(temp_pdf_path) # Clean up the temporary file
  else: # Assume it's a local file path
    with open(pdf_path, "rb") as pdf_file:
      pdf_reader = PdfReader(pdf_file)
      for page in pdf_reader.pages:
        text += page.extract_text()
  return text

In [4]:
# Setting the parameters for text splitting
sent_text_splitter = SentenceTransformersTokenTextSplitter(
    chunk_overlap=10, # Overlap tokens for context continuity
    model_name='sentence-transformers/all-MiniLM-L6-v2', # Embedding model for tokenization
    tokens_per_chunk=100 # Chunk size (tunable for your use case)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Chunking the text into smaller pieces for better processing and retrieval
def chunk_text(text,file_name):
  chunks = []
  for chunk in sent_text_splitter.split_text(text):
    chunks.append({"content":chunk,
                   "metadata":{"filename":file_name}})
  return chunks

In [6]:
# Storing the chunks in ChromaDB for efficient retrieval based on semantic similarity
def store_in_chroma(chunks,persist_directory="./chroma_store"):
  texts = [c["content"] for c in chunks]
  metadatas = [c["metadata"] for c in chunks]
  db = Chroma.from_texts(texts, embedding_model,metadatas=metadatas, persist_directory=persist_directory)
  return db

In [8]:
# Initializing the embedding model for converting text chunks into vector representations
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
pdf_path = "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"
filename = pdf_path.split("/")[-1]
text = extract_pdf_content(pdf_path)
chunks = chunk_text(text,filename)
db = store_in_chroma(chunks)

In [10]:
# Basic retrieval function to get relevant chunks from ChromaDB based on query similarity
def search_chroma(query,db,top_k=5):
  results = db.similarity_search(query,k=top_k)
  chunks = [{"content":d.page_content,"metadata":d.metadata} for d in results]
  return chunks

In [11]:
# --- Configure Gemini API Key ---
# Securely load your Google Gemini API key from Colab userdata.
# Use Case: Keeps credentials safe and enables authenticated LLM access.
import google.generativeai as genai # Correct import for genai.configure
from google.colab import userdata
api_key = userdata.get('G_API')
genai.configure(api_key=api_key)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [26]:
# Keyword search function for retrieving relevant chunks based on query words
def keyword_search(query, chunks):

    results = [] # Initialize an empty list to store scores and chunks

    # Convert the query to lowercase and split into individual words
    query_words = query.lower().split()

    # Iterate through each chunk in the provided list of chunks
    for chunk in chunks:

        # Convert the content of the current chunk to lowercase for case-insensitive matching
        text = chunk["content"].lower()

        # Calculate a score based on how many query words are present in the chunk's content
        score = sum(
            word in text # Check if each query word is in the chunk's text
            for word in query_words
        )

        # If the chunk contains at least one query word, add it to the results with its score
        if score > 0:
            results.append(
                (score, chunk) # Store as a tuple: (score, chunk_dictionary)
            )

    # Sort the results in descending order based on the score (first element of the tuple)
    results.sort(key=lambda x: x[0], reverse=True)

    # Return the top 5 chunks (without their scores), extracting only the chunk dictionaries
    return [
        r[1] # Extract the chunk dictionary from the (score, chunk) tuple
        for r in results[:5] # Get only the top 5 results
    ]

In [27]:
# Combines vector similarity search and keyword search for hybrid retrieval
def hybrid_retrieve(
    query, # The search query string
    db, # The ChromaDB instance for vector search
    all_chunks, # All available chunks for keyword search
    top_k=5 # The number of top results to return
):

    # Perform vector similarity search using ChromaDB
    vector_results = search_chroma(
        query,
        db,
        top_k
    )

    # Perform keyword search on all available chunks
    keyword_results = keyword_search(
        query,
        all_chunks
    )

    # Initialize a dictionary to store combined unique chunks, using chunk content as keys
    combined = {}

    # Iterate through both vector and keyword results to combine them
    for chunk in (
        vector_results + # Start with results from vector search
        keyword_results # Add results from keyword search
    ):

        # Add the chunk to the combined dictionary. If a chunk content already exists,
        # it will be overwritten (ensuring uniqueness based on content).
        combined[
            chunk["content"]
        ] = chunk

    # Convert the dictionary values (unique chunks) to a list and return the top_k results
    return list(
        combined.values()
    )[:top_k]

In [28]:
# Defines a hybrid RAG (Retrieval-Augmented Generation) function to answer queries
def hybrid_rag_answer(
    query, # The user's question
    db, # The ChromaDB instance for vector search
    all_chunks # All available text chunks for keyword search
):

    # Retrieve relevant chunks using a hybrid approach (vector + keyword search)
    chunks = hybrid_retrieve(
        query,
        db,
        all_chunks
    )

    # Combine the content of the retrieved chunks to form the context for the LLM
    context = "\n\n".join(
        [c["content"] for c in chunks]
    )

    # Construct the prompt for the generative model, including context and the question
    prompt = f"""
    Context:

    {context}

    Question:
    {query}

    Answer:
    """

    # Initialize the generative AI model (Gemini-2.5-flash in this case)
    model = genai.GenerativeModel(
        "models/gemini-2.5-flash"
    )

    # Generate a response based on the constructed prompt
    response = model.generate_content(
        prompt
    )

    # Return the text content of the generated response
    return response.text

In [29]:
query = """
What is attention and
how does it improve
transformers?
"""

In [30]:
hybrid_rag_answer(
    query,
    db,
    chunks
)

'Based on the provided text:\n\n**What is attention?**\nAttention is a mechanism that allows a model to "attend over" or focus on different parts of an input sequence (or internal representations) to draw global dependencies. For instance, in "encoder-decoder attention," it allows every position in the decoder to attend over all positions in the input sequence, establishing relationships between input and output. It helps the model weigh the importance of various parts of the sequence.\n\n**How does it improve Transformers?**\nAttention significantly improves Transformers by:\n\n1.  **Enabling Learning of Long-Range Dependencies:** It reduces the number of operations required to relate signals from two arbitrary input or output positions to a **constant**, irrespective of their distance. This makes it significantly easier to learn dependencies between distant positions compared to models like convs2s (linear growth) or bytenet (logarithmic growth).\n2.  **Facilitating Parallelization:*